<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/bioassay/bioassay_full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bioassay: a compact Bayesian workflow

**Short Bayesian course — worked example**

Four groups of five animals were exposed to different doses and the number of deaths was recorded. We will use a binomial logistic model to ask:

1. How does mortality change with dose?
2. What dose gives a 50% mortality probability — the **LD50**?

The statistical workflow is the point of the example:

$
\text{data}
\rightarrow
\text{model}
\rightarrow
\text{prior predictive}
\rightarrow
\text{fit and diagnose}
\rightarrow
\text{scientific quantity}
\rightarrow
\text{posterior predictive}.
$

The data and model follow Gelman & Vehtari, *Bayesian Workflow*, §3.5.

## 0. Setup

This notebook uses the current PyMC 6 / modular ArviZ stack. Plotting and statistical summaries are delegated to arviz-plots and arviz-stats instead of being reconstructed with NumPy and Matplotlib.

In [ ]:
%pip install -q pymc "arviz-plots[matplotlib]" arviz-stats

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260923
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## 1. Data

At dose $x_j$, we observe $y_j$ deaths among $n_j=5$ animals.

In [ ]:
dose = np.array([-0.86, -0.30, -0.05, 0.73])
n = np.array([5, 5, 5, 5])
deaths = np.array([0, 1, 3, 5])

bioassay = pd.DataFrame(
    {
        "dose_log_g_ml": dose,
        "animals": n,
        "deaths": deaths,
    }
)
bioassay["proportion_dead"] = bioassay["deaths"] / bioassay["animals"]
bioassay

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(dose, deaths / n, s=70)
ax.set(
    xlabel="Dose log(g/ml)",
    ylabel="Observed proportion dead",
    ylim=(-0.05, 1.05),
)
plt.show()

## 2. Generative model

For group $j$,

$$
y_j \sim \operatorname{Binomial}(n_j,p_j),
\qquad
\operatorname{logit}(p_j)=\alpha+\beta x_j.
$$

We use

$$
\alpha\sim N(0,5),
\qquad
\beta\sim \operatorname{HalfNormal}(5).
$$

The positive support of $\beta$ encodes the substantive assumption that mortality does not decrease as dose increases.

The mortality probability $p$ and LD50 are retained as PyMC deterministic quantities because we use them later. The expected number of deaths is calculated only when it becomes useful for plotting.

In [ ]:
coords = {"dose_log_g_ml": dose}

with pm.Model(coords=coords) as model:
    dose_data = pm.Data("dose", dose, dims="dose_log_g_ml")
    n_data = pm.Data("n", n, dims="dose_log_g_ml")

    alpha = pm.Normal("alpha", mu=0, sigma=5)
    beta = pm.HalfNormal("beta", sigma=5)

    logit_p = alpha + beta * dose_data

    # Retain p because the later dose-response plots use the probability scale directly.
    p = pm.Deterministic(
        "p",
        pm.math.sigmoid(logit_p),
        dims="dose_log_g_ml",
    )

    ld50_log_g_ml = pm.Deterministic(
        "LD50_log_g_ml",
        -alpha / beta,
    )
    ld50_mg_ml = pm.Deterministic(
        "LD50_mg_ml",
        1000 * pm.math.exp(ld50_log_g_ml),
    )

    pm.Binomial(
        "deaths",
        n=n_data,
        logit_p=logit_p,
        observed=deaths,
        dims="dose_log_g_ml",
    )

## 3. Prior predictive check

Before seeing the observed death counts, ask what the model can generate.

> **What kinds of observations do these priors say are plausible?**

For this grouped binomial outcome, look at the full prior predictive distribution of deaths separately at each dose.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(
        draws=1000,
        var_names=["alpha", "beta", "deaths"],
        random_seed=RANDOM_SEED,
    )

In [ ]:
azp.plot_dist(
    prior,
    var_names=["deaths"],
    group="prior_predictive",
    sample_dims=["chain", "draw"],
    kind="hist",
    cols=["dose_log_g_ml"],
    visuals={
        "credible_interval": False,
        "point_estimate": False,
        "point_estimate_text": False,
    },
)

plt.gcf().supxlabel("Deaths out of 5")
plt.gcf().suptitle("Prior predictive deaths by dose (log(g/ml))")

Each panel corresponds to one dose. With five animals per dose, the outcome can only be 0, 1, …, 5 deaths. The broad coefficient priors imply a correspondingly broad set of possible observations.

## 4. Fit and diagnose

Now condition on the observations. PyMC returns an xarray DataTree containing the posterior and sampler diagnostics.

Before interpreting the model, inspect divergences, $\hat R$, effective sample sizes, and the traces.

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        nuts={"target_accept": 0.90},
        random_seed=RANDOM_SEED,
    )

In [ ]:
print("Divergences:", idata["sample_stats"]["diverging"].sum().item())

azs.summary(
    idata,
    var_names=["alpha", "beta"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(
    idata,
    var_names=["alpha", "beta"],
)

## 5. Posterior dose-response fit

Before moving to LD50, look at what the posterior says about the fitted relationship at the four observed doses.

Expected deaths were not needed in the generative model, so calculate them now from the posterior mortality probability.

In [ ]:
idata["posterior"]["expected_deaths"] = (
    idata["posterior"]["p"] * idata["constant_data"]["n"]
)

azp.plot_lm(
    idata,
    x="dose",
    y="expected_deaths",
    y_obs="deaths",
    group="posterior",
    plot_dim="dose_log_g_ml",
    ci_prob=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
    smooth=False,
    visuals={"observed_scatter": False},
)

plt.scatter(dose, deaths, zorder=3)
plt.xlabel("Dose log(g/ml)")
plt.ylabel("Deaths out of 5")

## 6. Scientific quantity: LD50

The LD50 is the dose for which $p=0.5$. Because $\operatorname{logit}(0.5)=0$,

$
0=\alpha+\beta\,LD50
\quad\Longrightarrow\quad
LD50=-\frac{\alpha}{\beta}.
$

Because LD50 was declared inside the model, it already has a posterior distribution. We can hand that distribution directly to ArviZ.

In [ ]:
azs.summary(
    idata,
    var_names=["LD50_log_g_ml", "LD50_mg_ml"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_dist(
    idata,
    var_names=["LD50_mg_ml"],
    point_estimate="median",
    ci_prob=0.90,
    ci_kind="hdi",
)

## 7. Posterior predictive check

A posterior distribution for parameters is not yet a check of the model. Generate new death counts at the **same four doses** from the fitted model.

These replications include both uncertainty about the dose-response relationship and binomial variability among groups of five animals.

In [ ]:
with model:
    pm.sample_posterior_predictive(
        idata,
        var_names=["deaths"],
        extend_inferencedata=True,
        random_seed=RANDOM_SEED,
    )

In [ ]:
azp.plot_ppc_interval(
    idata,
    var_names=["deaths"],
    ci_probs=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
)

plt.xticks(np.arange(len(dose)), [f"{value:g}" for value in dose])
plt.xlabel("Dose log(g/ml)")

The question is whether the observed counts look ordinary relative to datasets the fitted model can generate. With four groups this is necessarily a modest check, but the workflow carries directly to richer examples.

## 8. Use the fitted model: dose-response prediction

Now evaluate the fitted relationship on a dense dose grid. Change the model's `pm.Data` and ask PyMC for out-of-sample **predictions**. PyMC stores the mortality probability in the predictions group; we convert it to expected deaths only for plotting on the same scale as the observations.

In [ ]:
dose_grid = np.linspace(-1.0, 1.0, 101)

with model:
    pm.set_data(
        {
            "dose": dose_grid,
            "n": np.full(dose_grid.size, 5),
        },
        coords={"dose_log_g_ml": dose_grid},
    )

    pm.sample_posterior_predictive(
        idata,
        var_names=["p"],
        predictions=True,
        extend_inferencedata=True,
        random_seed=RANDOM_SEED,
    )

    # Restore the observed design after prediction.
    pm.set_data(
        {
            "dose": dose,
            "n": n,
        },
        coords={"dose_log_g_ml": dose},
    )

idata["predictions"]["expected_deaths"] = 5 * idata["predictions"]["p"]

In [ ]:
azp.plot_lm(
    idata,
    x="dose",
    y="expected_deaths",
    y_obs="deaths",
    group="predictions",
    plot_dim="dose_log_g_ml",
    ci_prob=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
    smooth=False,
    visuals={"observed_scatter": False},
)

plt.scatter(dose, deaths, zorder=3)
plt.xlabel("Dose log(g/ml)")
plt.ylabel("Deaths out of 5")

The regression-fit plot and this dense prediction use the same vertical scale: expected deaths among five animals. The posterior predictive check above is different: it also includes the extra binomial variability in the realized number of deaths.

## 9. What to carry forward

This example establishes a reusable workflow:

- specify a generative model;
- inspect implications of the priors on the observable scale;
- fit and diagnose before interpreting;
- retain scientifically meaningful derived quantities in the model;
- generate replicated data to check the model;
- use the fitted model for prediction at new predictor values.

### Optional explorations

1. Replace HalfNormal(5) for $\beta$ with Normal(0, 5). Does the data rule out a decreasing relationship?
2. Try narrower priors and rerun the **prior predictive check before fitting**.
3. Predict a new group of five animals at one chosen dose. Distinguish uncertainty about $p$ from uncertainty about the realized death count.

## Sources

- Gelman, A. & Vehtari, A. *Bayesian Workflow*, §3.5, “Bioassay case study.”
- Racine-Poon, A., Grieve, A. P., Fluhler, H., & Smith, A. F. M. (1986). “Bayesian Methods in Practice: Experiences in the Pharmaceutical Industry.” *Applied Statistics*, 35, 93–150.
- PyMC 6 documentation.
- ArviZ plots/stats documentation.